# 🧮 Taller 56 - Filtro de Kalman e Inferencia de Variables Ocultas

**Asignatura:** Computación Visual  
**Objetivo:** Implementar y analizar el Filtro de Kalman en escenarios 1D y 2D para estimar trayectorias reales (variables ocultas) a partir de mediciones con ruido gaussiano.

---

## 1. Importación de Librerías y Carga de Datos

Primero, importamos las librerías necesarias para el análisis, la visualización y el cálculo numérico. Cargamos los datos sintéticos generados previamente en los archivos CSV.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configurar estilo de graficación
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

# Cargar datos
df_1d = pd.read_csv('datos_1d.csv')
df_2d = pd.read_csv('datos_2d.csv')

print("Datos 1D (Primeras 5 filas):")
print(df_1d.head())
print("\nDatos 2D (Primeras 5 filas):")
print(df_2d.head())

## 2. Implementación del Filtro de Kalman 1D

El caso unidimensional modela un estado escalar (posición) sujeto a variaciones del proceso y ruido de medición. A continuación se define la clase `KalmanFilter1D`:

In [ ]:
class KalmanFilter1D:
    def __init__(self, x_init=0.0, P_init=1.0, Q=0.001, R=4.0):
        self.x_hat = x_init
        self.P = P_init
        self.Q = Q
        self.R = R
        
        self.x_hat_prior = x_init
        self.P_prior = P_init
        self.K = 0.0

    def predict(self):
        # Predicción del estado y la covarianza del error
        self.x_hat_prior = self.x_hat
        self.P_prior = self.P + self.Q
        return self.x_hat_prior, self.P_prior

    def update(self, z):
        # Ganancia de Kalman
        self.K = self.P_prior / (self.P_prior + self.R)
        # Actualización del estado
        self.x_hat = self.x_hat_prior + self.K * (z - self.x_hat_prior)
        # Actualización de la covarianza
        self.P = (1.0 - self.K) * self.P_prior
        return self.x_hat, self.P, self.K

### Ejecutar el Filtro de Kalman 1D y Evaluar Resultados

Ejecutamos el estimador sobre las mediciones observadas y calculamos el Error Cuadrático Medio (MSE) y la Raíz del Error Cuadrático Medio (RMSE) respecto al valor real.

In [ ]:
# Parámetros reales usados en la generación: q_std=0.5 (var = 0.25), r_std=2.0 (var = 4.0)
Q_optimal = 0.25
R_optimal = 4.0

kf1d = KalmanFilter1D(x_init=df_1d['observed_x'].iloc[0], P_init=1.0, Q=Q_optimal, R=R_optimal)

estimates_1d = []
for z in df_1d['observed_x']:
    kf1d.predict()
    x_est, _, _ = kf1d.update(z)
    estimates_1d.append(x_est)

df_1d['estimated_x'] = estimates_1d

# Calcular métricas
real_1d = df_1d['real_x'].values
observed_1d = df_1d['observed_x'].values
predicted_1d = df_1d['estimated_x'].values

rmse_sensor = np.sqrt(np.mean((real_1d - observed_1d)**2))
rmse_kalman = np.sqrt(np.mean((real_1d - predicted_1d)**2))
improvement = (rmse_sensor - rmse_kalman) / rmse_sensor * 100

print(f"RMSE Sensor vs Real: {rmse_sensor:.4f}")
print(f"RMSE Kalman vs Real: {rmse_kalman:.4f}")
print(f"Mejora del filtro: {improvement:.2f}%")

# Graficar
plt.figure(figsize=(12, 6))
plt.plot(df_1d['step'], real_1d, label='Real (Variable Oculta)', color='#2c3e50', linestyle='--', linewidth=2)
plt.scatter(df_1d['step'], observed_1d, label='Medido (Sensor Ruidoso)', color='#e74c3c', alpha=0.5, s=20)
plt.plot(df_1d['step'], predicted_1d, label='Estimado (Kalman)', color='#2ecc71', linewidth=2.5)
plt.title('Estimación de Kalman 1D', fontsize=14, fontweight='bold')
plt.xlabel('Paso de Tiempo')
plt.ylabel('Posición')
plt.legend()
plt.show()

### Experimento de Sensibilidad: El Rol de la Relación Q / R

La relación $Q/R$ determina cuánta confianza deposita el filtro en el modelo del proceso (predicción) contra el sensor de medición (corrección).
- Si $Q \ll R$ (confianza alta en la predicción / sensor malo), el filtro suaviza fuertemente el ruido pero responde lentamente a cambios rápidos.
- Si $Q \gg R$ (confianza alta en la medición / modelo de proceso incierto), el filtro sigue de cerca las mediciones, filtrando muy poco el ruido.

In [ ]:
# Caso A: Q pequeño (Suavizado extremo / Inercia alta)
kf_smooth = KalmanFilter1D(x_init=observed_1d[0], Q=0.001, R=4.0)
# Caso B: Q grande (Seguimiento rápido / Filtrado pobre)
kf_noisy = KalmanFilter1D(x_init=observed_1d[0], Q=2.0, R=4.0)

est_smooth = []
est_noisy = []

for z in observed_1d:
    kf_smooth.predict()
    x_s, _, _ = kf_smooth.update(z)
    est_smooth.append(x_s)
    
    kf_noisy.predict()
    x_n, _, _ = kf_noisy.update(z)
    est_noisy.append(x_n)

plt.figure(figsize=(12, 6))
plt.plot(df_1d['step'], real_1d, 'k--', label='Real (Oculto)', alpha=0.7)
plt.scatter(df_1d['step'], observed_1d, color='red', alpha=0.15, s=15, label='Mediciones')
plt.plot(df_1d['step'], est_smooth, color='#9b59b6', label='Q/R muy bajo (Confía en predicción)', linewidth=2)
plt.plot(df_1d['step'], predicted_1d, color='#2ecc71', label='Q/R óptimo (Tuned)', linewidth=2)
plt.plot(df_1d['step'], est_noisy, color='#f1c40f', label='Q/R muy alto (Confía en sensor)', linewidth=1.5)
plt.title('Análisis de Sensibilidad: Variación de Q/R', fontsize=14, fontweight='bold')
plt.xlabel('Paso de Tiempo')
plt.ylabel('Posición')
plt.legend()
plt.show()

## 3. Implementación del Filtro de Kalman 2D

En 2D, representamos el estado por un vector de dimensión 4:
$$\mathbf{x} = [x, y, v_x, v_y]^T$$
Observamos únicamente la posición $(x, y)$, por lo tanto las velocidades $(v_x, v_y)$ actúan como variables completamente ocultas que serán inferidas dinámicamente por el filtro de Kalman.

In [ ]:
class KalmanFilter2D:
    def __init__(self, x_init=0.0, y_init=0.0, vx_init=0.0, vy_init=0.0, 
                 P_diag=[1.0, 1.0, 10.0, 10.0], dt=1.0, sigma_a=0.1, sigma_z=2.0):
        self.dt = dt
        self.x = np.array([[x_init], [y_init], [vx_init], [vy_init]], dtype=float)
        self.P = np.diag(P_diag).astype(float)
        
        # Matriz de transición de estado F
        self.F = np.array([[1.0, 0.0, dt,  0.0],
                           [0.0, 1.0, 0.0, dt ],
                           [0.0, 0.0, 1.0, 0.0],
                           [0.0, 0.0, 0.0, 1.0]], dtype=float)
        
        # Matriz de observación H
        self.H = np.array([[1.0, 0.0, 0.0, 0.0],
                           [0.0, 1.0, 0.0, 0.0]], dtype=float)
        
        # Matriz de ruido de proceso Q (Modelo de Aceleración Aleatoria)
        dt2, dt3, dt4 = dt**2, dt**3, dt**4
        self.Q = np.array([[dt4/4.0,   0.0,   dt3/2.0,   0.0],
                           [0.0,     dt4/4.0,   0.0,   dt3/2.0],
                           [dt3/2.0,   0.0,    dt2,      0.0],
                           [0.0,     dt3/2.0,   0.0,     dt2]], dtype=float) * (sigma_a**2)
        
        # Matriz de ruido de medición R
        self.R = np.eye(2, dtype=float) * (sigma_z**2)
        self.I = np.eye(4, dtype=float)
        
        self.x_prior = np.copy(self.x)
        self.P_prior = np.copy(self.P)

    def predict(self):
        self.x_prior = np.dot(self.F, self.x)
        self.P_prior = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q
        return self.x_prior, self.P_prior

    def update(self, z):
        z = np.array(z, dtype=float).reshape(2, 1)
        y = z - np.dot(self.H, self.x_prior)
        S = np.dot(np.dot(self.H, self.P_prior), self.H.T) + self.R
        K = np.dot(np.dot(self.P_prior, self.H.T), np.linalg.inv(S))
        self.x = self.x_prior + np.dot(K, y)
        self.P = np.dot((self.I - np.dot(K, self.H)), self.P_prior)
        return self.x, self.P, K

### Ejecutar el Filtro de Kalman 2D

In [ ]:
dt = 0.5
sigma_a = 0.3
sigma_z = 2.5

kf2d = KalmanFilter2D(
    x_init=df_2d['observed_x'].iloc[0],
    y_init=df_2d['observed_y'].iloc[0],
    vx_init=2.0, vy_init=1.5,
    P_diag=[10.0, 10.0, 10.0, 10.0],
    dt=dt,
    sigma_a=sigma_a,
    sigma_z=sigma_z
)

est_x, est_y, est_vx, est_vy = [], [], [], []

for ox, oy in zip(df_2d['observed_x'], df_2d['observed_y']):
    kf2d.predict()
    state, _, _ = kf2d.update([ox, oy])
    est_x.append(state[0, 0])
    est_y.append(state[1, 0])
    est_vx.append(state[2, 0])
    est_vy.append(state[3, 0])

df_2d['estimated_x'] = est_x
df_2d['estimated_y'] = est_y
df_2d['estimated_vx'] = est_vx
df_2d['estimated_vy'] = est_vy

# Calcular métricas de error de posición
errors_obs_2d = np.sqrt((df_2d['real_x'] - df_2d['observed_x'])**2 + (df_2d['real_y'] - df_2d['observed_y'])**2)
errors_est_2d = np.sqrt((df_2d['real_x'] - df_2d['estimated_x'])**2 + (df_2d['real_y'] - df_2d['estimated_y'])**2)

rmse_obs_2d = np.sqrt(np.mean(errors_obs_2d**2))
rmse_est_2d = np.sqrt(np.mean(errors_est_2d**2))
imp_2d = (rmse_obs_2d - rmse_est_2d) / rmse_obs_2d * 100

print(f"RMSE de posición de sensor: {rmse_obs_2d:.4f}")
print(f"RMSE de posición estimación Kalman: {rmse_est_2d:.4f}")
print(f"Mejora del error en 2D: {imp_2d:.2f}%")

### Visualización de la Trayectoria en Plano X-Y

Graficamos la trayectoria en el espacio bidimensional. Observamos cómo la línea estimada del Filtro de Kalman limpia el ruido caótico del sensor y reconstruye una trayectoria suave muy cercana a la real.

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(df_2d['real_x'], df_2d['real_y'], label='Trayectoria Real (Oculta)', color='#34495e', linestyle='--', linewidth=2)
plt.scatter(df_2d['observed_x'], df_2d['observed_y'], label='Mediciones de Posición (Sensor)', color='#e74c3c', alpha=0.35, s=25)
plt.plot(df_2d['estimated_x'], df_2d['estimated_y'], label='Trayectoria Estimada (Kalman)', color='#3498db', linewidth=2.5)
plt.scatter(df_2d['real_x'].iloc[0], df_2d['real_y'].iloc[0], color='black', s=80, zorder=5, label='Inicio')
plt.scatter(df_2d['real_x'].iloc[-1], df_2d['real_y'].iloc[-1], color='purple', marker='X', s=100, zorder=5, label='Fin')

plt.title('Filtro de Kalman 2D - Seguimiento en Plano X-Y', fontsize=14, fontweight='bold')
plt.xlabel('Posición X')
plt.ylabel('Posición Y')
plt.legend()
plt.axis('equal')
plt.show()

## 4. Inferencia de Variables Ocultas (Velocidades Vx, Vy)

Uno de los mayores beneficios del filtro de Kalman es su capacidad de estimar variables que no medimos directamente (variables latentes u ocultas).
En el caso 2D, el sensor de medición de posición **nunca mide la velocidad**, pero el filtro infiere dinámicamente $v_x$ y $v_y$ gracias a la matriz de transición de estado y el modelo físico interno de movimiento.

In [ ]:
plt.figure(figsize=(14, 6))

# Gráfico para Velocidad en X
plt.subplot(1, 2, 1)
plt.plot(df_2d['step'], df_2d['real_vx'], label='Velocidad Vx Real', color='#2c3e50', linestyle='--', linewidth=2)
plt.plot(df_2d['step'], df_2d['estimated_vx'], label='Vx Estimada por Kalman', color='#e67e22', linewidth=2.5)
plt.title('Inferencia de Velocidad Vx', fontsize=12, fontweight='bold')
plt.xlabel('Paso de Tiempo')
plt.ylabel('Velocidad Vx')
plt.legend()

# Gráfico para Velocidad en Y
plt.subplot(1, 2, 2)
plt.plot(df_2d['step'], df_2d['real_vy'], label='Velocidad Vy Real', color='#2c3e50', linestyle='--', linewidth=2)
plt.plot(df_2d['step'], df_2d['estimated_vy'], label='Vy Estimada por Kalman', color='#9b59b6', linewidth=2.5)
plt.title('Inferencia de Velocidad Vy', fontsize=12, fontweight='bold')
plt.xlabel('Paso de Tiempo')
plt.ylabel('Velocidad Vy')
plt.legend()

plt.tight_layout()
plt.show()

### Análisis del Error en Velocidades

Calculamos las desviaciones absolutas de las velocidades estimadas que no fueron medidas:

In [ ]:
rmse_vx = np.sqrt(np.mean((df_2d['real_vx'] - df_2d['estimated_vx'])**2))
rmse_vy = np.sqrt(np.mean((df_2d['real_vy'] - df_2d['estimated_vy'])**2))

print(f"RMSE en Velocidad Vx (Oculta): {rmse_vx:.4f}")
print(f"RMSE en Velocidad Vy (Oculta): {rmse_vy:.4f}")

---  
## 5. Conclusiones y Aprendizajes

1. **Reducción Efectiva de Ruido**: El filtro redujo el error de medición del sensor en aproximadamente un **60%**, tanto en el caso 1D como en el 2D.
2. **Inferencia de Variables No Observadas**: Pudimos estimar con gran precisión la velocidad del objeto en ambas direcciones (X e Y), a pesar de que el sensor solo registraba posiciones espaciales. Esto demuestra la potencia del Filtro de Kalman para inferir variables de estado ocultas combinando observaciones parciales con modelos cinemáticos físicos.
3. **Tuning del Filtro**: Se demostró de manera práctica la importancia de la relación $Q/R$. Configurar correctamente los ruidos de proceso y de medición es crítico para evitar retardos excesivos (fase tardía) o el paso descontrolado de ruido del sensor al estado estimado.